In [34]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import (
    normalized_mutual_info_score,
    adjusted_rand_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [35]:
RES = "sslpa_tx_lcc_manual_labels.csv"
LABELS_CSV = os.path.expanduser(
    "~/stellar-clustering/publication/labeled-data/normalization/labels_mapped_normalized.csv"
)
OUT_CSV = "lpa_evaluation.csv"

In [36]:
lpa_df = pd.read_csv(RES)
print(lpa_df.head(1))

lpa_df = lpa_df.rename(columns={'node': 'account_id', 'label': 'community'})
print(lpa_df.head(1))

   node         label
0     1  UltraCapital
   account_id     community
0           1  UltraCapital


In [37]:

labels_df = pd.read_csv(LABELS_CSV)
print(labels_df.head(1))



   account_id  name
0      563998  SCAM


In [38]:
lpa_df["account_id"] = lpa_df["account_id"].astype(str)
labels_df["account_id"] = labels_df["account_id"].astype(str)

merged_df = lpa_df.merge(labels_df, on="account_id", how="inner")

In [39]:
true_labels = merged_df["name"].astype(str).values
pred_clusters = merged_df["community"].astype(str).values

In [40]:
def compute_purity(true_labels, pred_labels):
    conf_matrix = confusion_matrix(true_labels, pred_labels)
    purity = np.sum(np.max(conf_matrix, axis=0)) / np.sum(conf_matrix)
    return purity

In [41]:
df_tmp = merged_df[["community", "name"]].copy()
majority_map = (
    df_tmp.groupby("community")["name"]
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)
pred_mapped = merged_df["community"].map(majority_map).astype(str).values


In [42]:

row = {
    "n_samples": len(true_labels),
    "true_classes": len(np.unique(true_labels)),
    "pred_clusters": len(np.unique(pred_clusters)),
    "nmi": normalized_mutual_info_score(true_labels, pred_clusters),
    "ari": adjusted_rand_score(true_labels, pred_clusters),
    "purity": compute_purity(true_labels, pred_clusters),

    # classification-style metrics after majority-vote mapping
    "precision": precision_score(true_labels, pred_mapped, average="weighted", zero_division=0),
    "recall": recall_score(true_labels, pred_mapped, average="weighted", zero_division=0),
    "f1": f1_score(true_labels, pred_mapped, average="weighted", zero_division=0),
}

In [43]:
out_df = pd.DataFrame([row])
out_df.to_csv(OUT_CSV, index=False)
display(out_df)

,n_samples,true_classes,pred_clusters,nmi,ari,purity,precision,recall,f1
0,8336,212,212,1.0,1.0,1.0,1.0,1.0,1.0


In [44]:
# 1) Are community and name literally the same (or almost)?
print((merged_df["community"].astype(str) == merged_df["name"].astype(str)).mean())

# 2) How many unique values in each?
print("unique community:", merged_df["community"].nunique())
print("unique name:", merged_df["name"].nunique())

# 3) Cross-tab a sample: does each community map to exactly one name and vice versa?
ct = pd.crosstab(merged_df["community"].astype(str), merged_df["name"].astype(str))
print("max in each community / total in community (first 10):")
print((ct.max(axis=1) / ct.sum(axis=1)).sort_values(ascending=False).head(10))
print("how many communities are 100% pure:", ((ct.max(axis=1) == ct.sum(axis=1))).sum(), "out of", ct.shape[0])


1.0
unique community: 212
unique name: 212
max in each community / total in community (first 10):
community
AADC                    1.0
R2B issuance account    1.0
Reddit Photons          1.0
Repocoin                1.0
RippleFox               1.0
SC AM                   1.0
SCAM                    1.0
SDF                     1.0
STEMchain               1.0
SciConLab               1.0
dtype: float64
how many communities are 100% pure: 212 out of 212


In [45]:
print(merged_df[["account_id","community","name"]].head(20))
print("equality rate:", (merged_df["community"] == merged_df["name"]).mean())
print("unique community sample:", merged_df["community"].unique()[:20])
print("unique name sample:", merged_df["name"].unique()[:20])


   account_id     community          name
0       55090  UltraCapital  UltraCapital
1         178       TMM bot       TMM bot
2        2955          AQUA          AQUA
3         270          SCAM          SCAM
4        2197          SCAM          SCAM
5       22229          SCAM          SCAM
6       22565          SCAM          SCAM
7      457250          SCAM          SCAM
8        4116        Uphold        Uphold
9         237        GateIO        GateIO
10       4970           OKX           OKX
11       4997          MEXC          MEXC
12     113672        Lobstr        Lobstr
13       6081       Binance       Binance
14      12223       Binance       Binance
15        403          AQUA          AQUA
16       2071      Coinbase      Coinbase
17    2059259       TMM bot       TMM bot
18        888       Binance       Binance
19       4357          SCAM          SCAM
equality rate: 1.0
unique community sample: ['UltraCapital' 'TMM bot' 'AQUA' 'SCAM' 'Uphold' 'GateIO' 'OKX' 'MEXC'
 'L